In [0]:
%python
dbutils.widgets.text("catalog_name" ,"dev")
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:

create or replace table identifier(:catalog_name || '.bronze.customer') as 

select * ,_metadata.file_path as filepath , current_timestamp() as _ingestedat   from samples.tpch.customer

In [0]:

create or replace table identifier(:catalog_name || '.bronze.lineitem') as 

select * ,_metadata.file_path as filepath , current_timestamp() as _ingestedat   from samples.tpch.lineitem

In [0]:

create or replace table identifier(:catalog_name || '.bronze.nation') as 

select * ,_metadata.file_path as filepath , current_timestamp() as _ingestedat   from samples.tpch.nation

In [0]:

create or replace table identifier(:catalog_name || '.bronze.orders') as 

select * ,_metadata.file_path as filepath , current_timestamp() as _ingestedat   from samples.tpch.orders

In [0]:

create or replace table identifier(:catalog_name || '.bronze.part') as 

select * ,_metadata.file_path as filepath , current_timestamp() as _ingestedat   from samples.tpch.part

In [0]:

create or replace table identifier(:catalog_name || '.bronze.partsupp') as 

select * ,_metadata.file_path as filepath , current_timestamp() as _ingestedat   from samples.tpch.partsupp

In [0]:

create or replace table identifier(:catalog_name || '.bronze.supplier') as 

select * ,_metadata.file_path as filepath , current_timestamp() as _ingestedat   from samples.tpch.supplier

In [0]:
-- Verify row counts: all bronze tables vs their source samples.tpch tables
select 'customer' as table_name,
       b.cnt as bronze_rows,
       s.cnt as source_rows,
       case when b.cnt = s.cnt then 'MATCH' else 'MISMATCH' end as status
from (select count(*) as cnt from identifier(:catalog_name || '.bronze.customer')) b
cross join (select count(*) as cnt from samples.tpch.customer) s

union all
select 'lineitem',
       (select count(*) from identifier(:catalog_name || '.bronze.lineitem')),
       (select count(*) from samples.tpch.lineitem),
       case when (select count(*) from identifier(:catalog_name || '.bronze.lineitem')) =
                 (select count(*) from samples.tpch.lineitem) then 'MATCH' else 'MISMATCH' end

union all
select 'nation',
       (select count(*) from identifier(:catalog_name || '.bronze.nation')),
       (select count(*) from samples.tpch.nation),
       case when (select count(*) from identifier(:catalog_name || '.bronze.nation')) =
                 (select count(*) from samples.tpch.nation) then 'MATCH' else 'MISMATCH' end

union all
select 'orders',
       (select count(*) from identifier(:catalog_name || '.bronze.orders')),
       (select count(*) from samples.tpch.orders),
       case when (select count(*) from identifier(:catalog_name || '.bronze.orders')) =
                 (select count(*) from samples.tpch.orders) then 'MATCH' else 'MISMATCH' end

union all
select 'part',
       (select count(*) from identifier(:catalog_name || '.bronze.part')),
       (select count(*) from samples.tpch.part),
       case when (select count(*) from identifier(:catalog_name || '.bronze.part')) =
                 (select count(*) from samples.tpch.part) then 'MATCH' else 'MISMATCH' end

union all
select 'partsupp',
       (select count(*) from identifier(:catalog_name || '.bronze.partsupp')),
       (select count(*) from samples.tpch.partsupp),
       case when (select count(*) from identifier(:catalog_name || '.bronze.partsupp')) =
                 (select count(*) from samples.tpch.partsupp) then 'MATCH' else 'MISMATCH' end

union all
select 'supplier',
       (select count(*) from identifier(:catalog_name || '.bronze.supplier')),
       (select count(*) from samples.tpch.supplier),
       case when (select count(*) from identifier(:catalog_name || '.bronze.supplier')) =
                 (select count(*) from samples.tpch.supplier) then 'MATCH' else 'MISMATCH' end

order by table_name

In [0]:
%python
# ──────────────────────────────────────────
# BACKFILL: region table was missing from bronze.
# It's needed because nation has a FK to region.
# ──────────────────────────────────────────
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog_name}.bronze.region AS
    SELECT *, _metadata.file_path AS filepath, current_timestamp() AS _ingestedat
    FROM samples.tpch.region
""")
print("✅ Backfilled dev.bronze.region")